# A Regression Use Case for Feast: Diabetes Progression

This notebook applies the same online/offline feature store pattern from the session — this time to a **regression** problem instead of classification, to show the pattern is task-agnostic.

**Dataset**: the classic scikit-learn Diabetes dataset — 442 patients, 10 baseline measurements (age, sex, BMI, blood pressure, six blood serum values), and a continuous target: a quantitative measure of disease progression one year after baseline. This is a real, widely-used regression benchmark, not synthetic data.

## Plan

1. Load the dataset and engineer one interaction feature on top of the raw measurements
2. Define a Feast feature repository: `Entity`, `FeatureView`, offline (Parquet) and online (SQLite) stores
3. Register the definitions with Feast
4. Fetch **historical** features (offline store) and train a regression model
5. **Materialize** features into the online store
6. Fetch **online** features to simulate a live prediction request
7. Serve predictions using the online features
8. Run a consistency check — offline vs. online values for the same patients — the same core feature store guarantee from the main exercise, now shown on a second, different dataset and task

## Step 0: Setup

In [1]:
# Install dependencies (only needs to run once per environment)
%pip install feast scikit-learn --quiet

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(0)

REPO_PATH = "feature_repo"
os.makedirs(f"{REPO_PATH}/data", exist_ok=True)

print("Feature repo folder ready at:", os.path.abspath(REPO_PATH))

Feature repo folder ready at: C:\Shridhar\Study\BITS\Week 9\Demo\feature_repo


## Step 1: Load Data and Engineer a Feature

The raw dataset arrives with 10 baseline measurements already mean-centered and scaled (a common real-world situation: upstream teams sometimes hand you pre-processed features). We add one engineered feature of our own — an interaction term between blood pressure and BMI, since the combination of the two is clinically more informative than either alone.

As before, Feast needs an **entity id** and an **event timestamp** per row, which this dataset doesn't naturally have, so we synthesize them.

In [4]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
df = diabetes.frame.copy()
df["patient_id"] = df.index

# Engineered feature: blood pressure * BMI interaction
df["bp_bmi_interaction"] = df["bp"] * df["bmi"]

# Synthetic timestamps: pretend records were collected 10 minutes apart
start = datetime.now() - timedelta(days=30)
df["event_timestamp"] = [start + timedelta(minutes=10 * i) for i in range(len(df))]
df["created"] = df["event_timestamp"]

print(df.shape)
df.head()

(442, 15)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,patient_id,bp_bmi_interaction,event_timestamp,created
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,0,0.001349,2026-06-15 20:48:53.469639,2026-06-15 20:48:53.469639
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,1,0.001355,2026-06-15 20:58:53.469639,2026-06-15 20:58:53.469639
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,2,-0.000252,2026-06-15 21:08:53.469639,2026-06-15 21:08:53.469639
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,3,0.000425,2026-06-15 21:18:53.469639,2026-06-15 21:18:53.469639
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,4,-0.000796,2026-06-15 21:28:53.469639,2026-06-15 21:28:53.469639


In [5]:
# Save as Parquet -- this file plays the role of our offline store's data source
df.to_parquet(f"{REPO_PATH}/data/diabetes_features.parquet", index=False)
print("Saved offline source parquet file.")

Saved offline source parquet file.


## Step 2: Define the Feature Repository

One `Entity` (`patient_id`), one `FeatureView` grouping all eleven numeric features (the ten original measurements plus our engineered interaction term), backed by the Parquet file we just wrote.

In [7]:
from datetime import timedelta
from feast import FeatureStore, Entity, FeatureView, Field, FileSource
from feast.types import Float32

patient = Entity(name="patient_id", description="Row identifier for a diabetes patient record")

diabetes_source = FileSource(
    path=os.path.abspath(f"{REPO_PATH}/data/diabetes_features.parquet"),
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

feature_names = ["age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6", "bp_bmi_interaction"]

diabetes_features_view = FeatureView(
    name="diabetes_features",
    entities=[patient],
    ttl=timedelta(days=365),
    schema=[Field(name=f, dtype=Float32) for f in feature_names],
    source=diabetes_source,
    online=True,
)

print("Entity and feature view defined:", len(feature_names), "features")

Entity and feature view defined: 11 features


C:\Users\galan\AppData\Local\Temp\ipykernel_20640\4041282492.py:5: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'patient_id'.
  patient = Entity(name="patient_id", description="Row identifier for a diabetes patient record")


In [8]:
# Write feature_store.yaml -- tells Feast where the registry and online store live
yaml_content = f"""project: diabetes_feature_store
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
"""

with open(f"{REPO_PATH}/feature_store.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

project: diabetes_feature_store
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3



In [9]:
# Register (apply) our definitions -- equivalent to `feast apply` from the CLI
store = FeatureStore(repo_path=REPO_PATH)
store.apply([patient, diabetes_features_view])

print("Registered feature views:")
for fv in store.list_feature_views():
    print(" -", fv.name, "->", len(fv.features), "features")

Registered feature views:
 - diabetes_features -> 11 features


## Step 3: Fetch Historical Features for Training

Point-in-time correct retrieval from the offline store, same as any classification use case — the label column (`target`, a continuous value here rather than a class) rides along in the entity dataframe.

In [16]:
entity_df = df[["patient_id", "event_timestamp", "target"]]

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[f"diabetes_features:{f}" for f in feature_names],
).to_df()

print("Training dataframe shape:", training_df.shape)
training_df.head()

Training dataframe shape: (442, 14)


,patient_id,event_timestamp,target,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,bp_bmi_interaction
0,0,2026-06-15 20:48:53.469639+00:00,151.0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,0.001349
1,1,2026-06-15 20:58:53.469639+00:00,75.0,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,0.001355
2,2,2026-06-15 21:08:53.469639+00:00,141.0,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,-0.000252
3,3,2026-06-15 21:18:53.469639+00:00,206.0,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,0.000425
4,4,2026-06-15 21:28:53.469639+00:00,135.0,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,-0.000796


## Step 4: Train a Regression Model

A `RandomForestRegressor` on the offline features. This is the same "offline store -> training set -> model" path as the classification exercise, just with a regression estimator and regression metrics (RMSE, R²) instead of accuracy.

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

X = training_df[feature_names]
y = training_df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse = mean_squared_error(y_test, preds) ** 0.5
r2 = r2_score(y_test, preds)

print(f"Test RMSE: {rmse:.2f}")
print(f"Test R^2:  {r2:.3f}")

Test RMSE: 53.53
Test R^2:  0.459


An R² in the 0.4–0.5 range is in line with published results for this dataset with a default random forest — it's a genuinely noisy, hard regression problem, which is realistic rather than a red flag. The point of this notebook isn't model performance; it's the feature store plumbing around the model.

## Step 5: Materialize Features into the Online Store

Push the latest feature values into the SQLite online store, exactly as a scheduled materialization job would in production.

In [21]:
store.materialize_incremental(end_date=datetime.now())
print("Materialization complete.")

Materializing 1 feature views to 2026-07-15 20:49:45+00:00 into the sqlite online store.

diabetes_features from 2025-07-15 15:19:45+00:00 to 2026-07-15 20:49:45+00:00:
Materialization complete.


## Step 6: Fetch Online Features and Serve a Prediction

Pick a handful of patients and fetch their features from the online store, exactly as a real-time inference API would given a `patient_id`.

In [25]:
sample_ids = df["patient_id"].sample(5, random_state=1).tolist()

online_features = store.get_online_features(
    features=[f"diabetes_features:{f}" for f in feature_names],
    entity_rows=[{"patient_id": sid} for sid in sample_ids],
).to_df()

online_features

C:\Users\galan\anaconda3\Lib\site-packages\faiss\loader.py:49: DeprecationWarning: numpy.core._multiarray_umath is deprecated and has been renamed to numpy._core._multiarray_umath. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core._multiarray_umath.__cpu_features__.
  from numpy.core._multiarray_umath import __cpu_features__
C:\Users\galan\anaconda3\Lib\site-packages\feast\infra\online_stores\milvus_online_store\milvus.py:100: UserWarning: Field name "vector_enabled" in "MilvusOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  class MilvusOnlineStoreConfig(FeastConfigBaseModel, VectorStoreConfig):


,patient_id,s6,s4,s3,s5,s1,s2,bmi,bp,age,bp_bmi_interaction,sex
0,246,-0.059067,-0.009972,0.056003,0.045067,0.079612,0.050982,-0.032073,-0.061892,0.041708,0.001985,-0.044642
1,425,-0.050783,-0.076395,0.022869,-0.020292,-0.100638,-0.112795,-0.040696,-0.081413,-0.078165,0.003313,-0.044642
2,293,-0.001078,0.000360,0.000779,-0.054540,0.020446,0.042527,0.092953,0.012680,-0.070900,0.001179,-0.044642
3,31,-0.042499,-0.076395,0.059685,-0.037129,-0.038720,-0.053610,-0.065486,-0.081413,-0.023677,0.005331,-0.044642
4,359,0.061054,0.034309,-0.024993,0.014821,0.006687,0.017475,0.005650,0.032201,0.038076,0.000182,0.050680


In [26]:
X_serve = online_features[feature_names]
predicted_progression = model.predict(X_serve)

true_vals = df[df["patient_id"].isin(sample_ids)][["patient_id", "target"]]

result = online_features[["patient_id"]].copy()
result["predicted_progression"] = predicted_progression.round(1)
result = result.merge(true_vals, on="patient_id")
result["abs_error"] = (result["predicted_progression"] - result["target"]).abs().round(1)

result

,patient_id,predicted_progression,target,abs_error
0,246,111.4,78.0,33.4
1,425,107.2,152.0,44.8
2,293,190.1,200.0,9.9
3,31,83.1,59.0,24.1
4,359,212.1,311.0,98.9


## Step 7: Consistency Check — Offline vs. Online

The same guarantee we verified in the classification exercise: features served online should match the features available offline for the same entity, since both were populated from the one shared `diabetes_features` definition.

In [28]:
latest_offline = df[df["patient_id"].isin(sample_ids)][["patient_id", "bp_bmi_interaction"]].rename(
    columns={"bp_bmi_interaction": "bp_bmi_interaction_offline"}
)

comparison = online_features[["patient_id", "bp_bmi_interaction"]].rename(
    columns={"bp_bmi_interaction": "bp_bmi_interaction_online"}
)
comparison = comparison.merge(latest_offline, on="patient_id")
comparison["difference"] = (
    comparison["bp_bmi_interaction_online"] - comparison["bp_bmi_interaction_offline"]
).abs()

comparison

,patient_id,bp_bmi_interaction_online,bp_bmi_interaction_offline,difference
0,246,0.001985,0.001985,1.143363e-10
1,425,0.003313,0.003313,2.854384e-11
2,293,0.001179,0.001179,5.519460e-11
3,31,0.005331,0.005331,2.289781e-11
4,359,0.000182,0.000182,6.080193e-12


`difference` at (or extremely near) zero for every patient confirms the pattern holds regardless of task type: one feature definition, materialized twice, read from two different code paths, still agrees.

## Wrap-Up

This notebook re-ran the full Feast pattern — `Entity` + `FeatureView`, offline Parquet source, online SQLite store, point-in-time historical retrieval, materialization, online serving — on a **regression** task instead of classification, using a real dataset (scikit-learn's Diabetes dataset) rather than synthetic data.

Nothing about the feature store layer changed between the classification and regression use cases: the entity, feature view, materialization call, and online lookup APIs are identical regardless of what kind of model consumes the features downstream. That task-agnosticism is exactly the point of separating "feature infrastructure" from "modeling" as two different concerns — which is the core idea this whole session has been building toward.